# Notebook 02 — Filtering and selecting Lyon restaurants

This notebook takes the ~2800 Lyon restaurants from notebook 01 and narrows
them down to 100 high-quality, diverse restaurants that will form the
base of the Connoisseur Companion agent.

Filtering strategy:
1. **Quality filter**: keep only restaurants with rating ≥ 4.0 and ≥ 50 reviews
2. **Completeness filter**: drop rows missing critical fields (name, cuisines, address)
3. **Diversity selection**: sample across cuisine types and price levels

The output is `data/processed/lyon_restaurants_selected.csv`.

In [1]:
"""Notebook 02 — Filtering and selecting Lyon restaurants."""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INPUT_CSV = DATA_PROCESSED / "lyon_restaurants_raw.csv"
OUTPUT_CSV = DATA_PROCESSED / "lyon_restaurants_selected.csv"

# Load the Lyon subset from notebook 01
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df):,} Lyon restaurants")
print(f"Columns: {df.shape[1]}")

Loaded 2,930 Lyon restaurants
Columns: 42


In [2]:
# Identify the key columns we'll work with. The dataset uses slightly different
# naming conventions across versions, so we look for likely names.

candidate_cols = {
    "name": ["restaurant_name", "name", "title"],
    "rating": ["avg_rating", "rating", "food_rating"],
    "reviews": ["total_reviews_count", "reviews_count", "n_reviews", "number_of_reviews"],
    "cuisines": ["cuisines", "cuisine_style", "cuisine"],
    "price": ["price_level", "price_range", "price"],
    "address": ["address", "street_address", "location"],
}

# Find which one actually exists
COLS = {}
for key, candidates in candidate_cols.items():
    for c in candidates:
        if c in df.columns:
            COLS[key] = c
            break
    else:
        COLS[key] = None
        print(f"⚠️ No column found for '{key}'")

print("\nResolved column names:")
for key, col in COLS.items():
    print(f"  {key:10} → {col}")


Resolved column names:
  name       → restaurant_name
  rating     → avg_rating
  reviews    → total_reviews_count
  cuisines   → cuisines
  price      → price_level
  address    → address


In [3]:
# Quality filter: rating ≥ 4.0 AND ≥ 50 reviews
MIN_RATING = 4.0
MIN_REVIEWS = 50

quality_df = df[
    (df[COLS["rating"]] >= MIN_RATING) &
    (df[COLS["reviews"]] >= MIN_REVIEWS)
].copy()

print(f"After quality filter: {len(quality_df):,} restaurants")
print(f"  (rating ≥ {MIN_RATING} AND reviews ≥ {MIN_REVIEWS})")

After quality filter: 819 restaurants
  (rating ≥ 4.0 AND reviews ≥ 50)


In [4]:
# Drop rows where critical fields are missing
critical_cols = [COLS["name"], COLS["cuisines"], COLS["address"]]
before = len(quality_df)

# Drop rows where any critical column is NaN
clean_df = quality_df.dropna(subset=critical_cols).copy()
after = len(clean_df)

print(f"Before completeness filter: {before:,}")
print(f"After completeness filter:  {after:,}")
print(f"Dropped: {before - after:,}")

Before completeness filter: 819
After completeness filter:  793
Dropped: 26


In [5]:
# Distribution of cuisines (each restaurant can have multiple cuisines)
# Split by comma and count
cuisines_series = (
    clean_df[COLS["cuisines"]]
    .str.split(",")
    .explode()
    .str.strip()
)

print("Top 20 cuisine types in our pool:")
print(cuisines_series.value_counts().head(20))

Top 20 cuisine types in our pool:
cuisines
French            508
European          319
Asian              92
Italian            78
Healthy            57
Mediterranean      43
Gastropub          42
Pizza              42
American           41
Bar                41
Wine Bar           40
Japanese           38
Contemporary       30
Fast food          27
Vietnamese         26
Pub                26
Sushi              24
Cafe               22
Middle Eastern     21
International      20
Name: count, dtype: int64


In [6]:
if COLS["price"]:
    print(f"Distribution of {COLS['price']}:")
    print(clean_df[COLS["price"]].value_counts())

Distribution of price_level:
price_level
€€-€€€    625
€         106
€€€€       62
Name: count, dtype: int64


In [7]:
# Extract the primary cuisine (the first one in the comma-separated list)
clean_df["primary_cuisine"] = (
    clean_df[COLS["cuisines"]]
    .str.split(",")
    .str[0]  # take the first one
    .str.strip()
)

print("Top 15 primary cuisines:")
print(clean_df["primary_cuisine"].value_counts().head(15))

Top 15 primary cuisines:
primary_cuisine
French       471
Italian       72
Japanese      35
Asian         29
American      22
Indian        15
Chinese       15
European      13
Lebanese       9
Bar            8
Fast food      7
Wine Bar       7
Gastropub      6
Healthy        6
Cafe           5
Name: count, dtype: int64


In [12]:
# Target: 100 restaurants total
TARGET_TOTAL = 150

# Take top N restaurants per primary cuisine, sorted by rating then reviews
def select_top_per_cuisine(df, top_n=5, min_per_cuisine=1):
    """
    For each primary cuisine, take the top N restaurants
    (sorted by rating desc, then reviews desc as tiebreaker).
    """
    selections = []
    for cuisine, group in df.groupby("primary_cuisine"):
        n = max(min_per_cuisine, min(top_n, len(group)))
        top = group.sort_values(
            [COLS["rating"], COLS["reviews"]],
            ascending=[False, False],
        ).head(n)
        selections.append(top)
    return pd.concat(selections)

# Start with 3 per cuisine
selected = select_top_per_cuisine(clean_df, top_n=5)
print(f"Selected after 'top 5 per cuisine': {len(selected)}")

Selected after 'top 5 per cuisine': 148


In [13]:
# Tune top_n to land near TARGET_TOTAL
for n in [1, 2, 3, 4, 5]:
    selected = select_top_per_cuisine(clean_df, top_n=n)
    print(f"top_n={n} → {len(selected)} restaurants")

top_n=1 → 50 restaurants
top_n=2 → 87 restaurants
top_n=3 → 114 restaurants
top_n=4 → 133 restaurants
top_n=5 → 148 restaurants


In [14]:
# Final selection
FINAL_TOP_N = 5  # adjust based on what gave you ~150

selected = select_top_per_cuisine(clean_df, top_n=FINAL_TOP_N)
selected = selected.reset_index(drop=True)

print(f"\nFinal selection: {len(selected)} restaurants")
print(f"\nBreakdown by primary cuisine:")
print(selected["primary_cuisine"].value_counts())


Final selection: 148 restaurants

Breakdown by primary cuisine:
primary_cuisine
American            5
Bar                 5
Cafe                5
Asian               5
Chinese             5
Healthy             5
Gastropub           5
French              5
Wine Bar            5
Lebanese            5
Italian             5
Japanese            5
Indian              5
European            5
Fast food           5
Moroccan            4
International       4
Brew Pub            4
Seafood             4
Mexican             3
Barbecue            3
African             3
Spanish             3
Middle Eastern      3
Pizza               3
Pub                 3
Contemporary        3
Armenian            2
Turkish             2
Peruvian            2
Irish               2
Latin               2
Cajun & Creole      2
Diner               2
Thai                2
Steakhouse          2
Street Food         2
Cuban               1
Argentinian         1
Brazilian           1
Caribbean           1
Colombian        

In [15]:
# Show key columns of the selection
preview_cols = [
    COLS["name"],
    "primary_cuisine",
    COLS["price"],
    COLS["rating"],
    COLS["reviews"],
]
preview_cols = [c for c in preview_cols if c]  # skip None

selected[preview_cols].sort_values(COLS["rating"], ascending=False).head(20)

,restaurant_name,primary_cuisine,price_level,avg_rating,total_reviews_count
11,Le Cambodia,Asian,€,5.0,56.0
62,Le Comptoir des Cousins,French,€€-€€€,5.0,486.0
60,Aromatic,French,€€-€€€,5.0,862.0
71,Le Neuvieme Art,Healthy,€€€€,5.0,840.0
87,Pasta & Basta,Italian,€€-€€€,5.0,82.0
88,MAX CALL,Italian,€,5.0,53.0
89,Via Barcatta,Italian,€€-€€€,5.0,50.0
92,Bentomania Honten,Japanese,€€-€€€,5.0,55.0
110,Mojgan,Middle Eastern,€€-€€€,5.0,168.0
101,Rose de Damas,Lebanese,€€-€€€,5.0,106.0


In [16]:
# Keep only useful columns for the rest of the pipeline
keep_cols = [
    COLS["name"],
    COLS["address"],
    "primary_cuisine",
    COLS["cuisines"],
    COLS["price"],
    COLS["rating"],
    COLS["reviews"],
]

# Add any other interesting columns if they exist
for c in ["top_tags", "awards", "special_diets", "features", "open_days_per_week"]:
    if c in selected.columns:
        keep_cols.append(c)

keep_cols = [c for c in keep_cols if c]
selected_final = selected[keep_cols].copy()

print(f"Final shape: {selected_final.shape}")
selected_final.head()

Final shape: (148, 12)


,restaurant_name,address,primary_cuisine,cuisines,price_level,avg_rating,total_reviews_count,top_tags,awards,special_diets,features,open_days_per_week
0,Lyon-Dakar,"227 rue de Crequi, 69003 Lyon France",African,African,€€-€€€,4.0,189.0,"Mid-range, African","Certificate of Excellence 2019, Certificate of...",NaN,NaN,5.0
1,Mattsam Restaurant Messob,"85 rue Massena, 69006 Lyon France",African,"African, Ethiopian",€€-€€€,4.0,140.0,"Mid-range, African, Ethiopian, Vegetarian Frie...","Travellers' Choice, Certificate of Excellence ...","Vegetarian Friendly, Vegan Options",NaN,NaN
2,La Mangue Amère,"7 rue du Jardin des Plantes, 69001 Lyon France",African,African,€€-€€€,4.0,91.0,"Mid-range, African","Travellers' Choice, Certificate of Excellence ...",NaN,NaN,7.0
3,Two Amigos (Ampère Victor Hugo),"1 rue Henri IV, 69002 Lyon France",American,"American, Mexican, Latin",€€-€€€,4.5,346.0,"Mid-range, Mexican, American, Latin","Travellers' Choice, Certificate of Excellence ...","Vegetarian Friendly, Vegan Options",NaN,7.0
4,Delicatessen,"6 rue Savy, 69001 Lyon France",American,"American, Barbecue",€€-€€€,4.5,186.0,"Mid-range, American, Barbecue","Travellers' Choice, Certificate of Excellence ...",NaN,NaN,5.0


In [17]:
selected_final.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved {len(selected_final)} restaurants to {OUTPUT_CSV}")
print(f"   File size: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

✅ Saved 148 restaurants to C:\Users\busar\Desktop\connoisseur\data\processed\lyon_restaurants_selected.csv
   File size: 44.3 KB
